# P2.4 Tune Gold Audit — Final

Independent strict validation and complete semantic branch audit for all 100 tune questions, including 21/21 detailed complex cases. No code cell reads the locked split.

In [1]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'vifinqa').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from vifinqa.utils.io import read_jsonl
from vifinqa.devset.p24 import validate_gold_records
from vifinqa.devset.p24_authoring_ext import P24ForensicTableLoader
from vifinqa.devset.p24_gold_audit import build_tune_gold_audit
from vifinqa.devset.p24_semantic_audit_v2 import build_complete_complex_semantic_audit
BUNDLE = ROOT / 'artifacts' / 'devset_p24'
gold = read_jsonl(BUNDLE / 'p24_tune_gold.final.jsonl')
specs = read_jsonl(BUNDLE / 'p24_tune_authoring.final.v2.jsonl')
questions = read_jsonl(BUNDLE / 'p24_tune_questions.jsonl')
LOCKED_OPENED = False
len(gold), len(specs), len(questions), LOCKED_OPENED

(100, 100, 100, False)

In [2]:
validation = validate_gold_records(gold, questions, 'tune',
    table_loader=P24ForensicTableLoader(ROOT / 'artifacts' / 'store'),
    require_complete=True)
audit = build_tune_gold_audit(gold, questions, specs)
semantic = build_complete_complex_semantic_audit(gold)
{**validation, 'gold_sha256': audit['gold_sha256'],
 'complex_semantic_checks': semantic['detailed_check_count'],
 'locked_opened': LOCKED_OPENED}

{'split': 'tune',
 'count': 100,
 'complete': True,
 'records_sha256': '8a3725276c4baafbb3ecbeaa3ea8f3bfcc488f16406e3994bd09b3a1a255d331',
 'gold_sha256': '8a3725276c4baafbb3ecbeaa3ea8f3bfcc488f16406e3994bd09b3a1a255d331',
 'complex_semantic_checks': 21,
 'locked_opened': False}

In [3]:
pd.DataFrame([audit['provenance']]).T.rename(columns={0: 'value'})

,value
evidence_total,578.0
evidence_unique_exact_cells,502.0
unique_tables,397.0
unique_reports,235.0
evidence_per_question_min,1.0
evidence_per_question_median,2.0
evidence_per_question_max,36.0


In [4]:
pd.DataFrame(audit['review_flags']).merge(
    pd.DataFrame(audit['records'])[['id', 'notes']], on='id', how='left')

,id,answer,output_type,reasons,notes
0,417,0.94,ratio,[large_evidence_graph],Rank the five food companies by CFO margin min...
1,447,128.64,ratio,"[ratio_magnitude_gt_100, large_evidence_graph]",Filter above-median 2024-2025 revenue growth; ...
2,570,-8.73,percentage_point,[large_evidence_graph],Rank increase in 365*average inventory/COGS fr...
3,615,224.44,percent,[percent_magnitude_gt_100],"Unfinished real estate, 2020 versus 2017; exac..."
4,732,902.43,ratio,[ratio_magnitude_gt_100],Parent-company short-term borrowings divided b...
5,893,58.69,number,[large_evidence_graph],Sum every current-year parent-company related-...


In [5]:
pd.DataFrame([{
    'id': item['id'], 'kind': item['kind'], 'answer': item['answer'],
    'selected': item.get('selected', item.get('selected_year', item.get('selected_base_year', ''))),
    'recomputed': item['recomputed'],
} for item in semantic['checks']])

,id,kind,answer,selected,recomputed
0,372,argmin_following_year_project,0.13,2023,0.128043
1,375,median_partition,24.94,,24.942107
2,383,count_after_first_with_dual_condition,2.00,,2.000000
3,397,filter_argmin_project,146.61,NVL,146.607441
4,417,argmax_project,0.94,MPC,0.941190
5,425,scenario_argmax_project,4.49,2024,4.494545
6,446,below_median_share,49.58,"[DBC, QNS, VNM]",49.582651
7,447,filter_aggregate_argmax_denominator,128.64,"[DBC, OGC, VNM]",128.642655
8,468,positive_pat_conditional_average,-4.50,"[HHV, VSC]",-4.501040
9,473,all_years_positive_margin_conditional_sum,206.67,"[HPG, HSG, MSR]",206.669458


In [6]:
assert validation['count'] == 100 and validation['complete']
assert semantic['count'] == semantic['detailed_check_count'] == 21
assert semantic['metadata_value_columns'] == []
assert all(semantic['duplicate_invariants'].values())
assert not LOCKED_OPENED
{'status': 'PASS', 'gold_sha256': audit['gold_sha256'], 'records': 100,
 'complex_semantic_checks': 21, 'locked_opened': LOCKED_OPENED}

{'status': 'PASS',
 'gold_sha256': '8a3725276c4baafbb3ecbeaa3ea8f3bfcc488f16406e3994bd09b3a1a255d331',
 'records': 100,
 'complex_semantic_checks': 21,
 'locked_opened': False}